In [353]:
import numpy as np
import pandas as pd

In [354]:
raw_data=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")

In [355]:
raw_data

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [356]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [357]:
ord3_col=['BsmtFinType1','BsmtFinType2']
ord_col=['BsmtQual','BsmtCond','HeatingQC','FireplaceQu','GarageCond','PoolQC','ExterQual',
         'ExterCond','KitchenQual']

In [358]:
quality_map = {
    'None': 0,
    'Po': 1,
    'Fa': 2,
    'TA': 3,
    'Gd': 4,
    'Ex': 5
}

bsmt_exposure_map = {
    'None': 0, 
    'No': 1,
    'Mn': 2,
    'Av': 3,
    'Gd': 4
}

bsmt_fin_map = {
    'None': 0,
    'Unf': 1,
    'LwQ': 2,
    'Rec': 3,
    'BLQ': 4,
    'ALQ': 5,
    'GLQ': 6
}

gar_map={
    'None': 0,
    'Unf': 1,
    'RFn': 2,
    'Fin': 3
}
centralair_map={
    'Y': 1,
    'N': 0
}
paveddrive_map={
    'Y': 2,
    'P': 1,
    'N': 0
}

In [359]:
def feature_engineering(df):

    df = df.copy()

    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['YearsToRemodel'] = df['YearRemodAdd'] - df['YearBuilt']
    df['Remodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)

    df.drop(
        columns=[
            'Id',
            'MoSold',
            'GarageCars',
            'GarageQual',
            'GarageYrBlt',
            'YrSold',
            'YearRemodAdd',
            'YearBuilt'
        ],
        inplace=True
    )

    return df

def ordinal_encode(df):

    df = df.copy()

    for col in ord_col:
        df[col] = df[col].fillna('None').map(quality_map)

    df['BsmtExposure'] = (
        df['BsmtExposure']
        .fillna('None')
        .map(bsmt_exposure_map)
    )

    for col in ord3_col:
        df[col] = (
            df[col]
            .fillna('None')
            .map(bsmt_fin_map)
        )

    df['GarageFinish'] = (
        df['GarageFinish']
        .fillna('None')
        .map(gar_map)
    )

    df['CentralAir'] = df['CentralAir'].map(centralair_map)

    df['PavedDrive'] = df['PavedDrive'].map(paveddrive_map)

    return df

In [360]:
raw_data = feature_engineering(raw_data)
raw_data = ordinal_encode(raw_data)

In [361]:
cat_cols = raw_data.select_dtypes(include=['object']).columns.tolist()
num_cols = raw_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

mod_col=['Electrical']

cat_cols=[col for col in cat_cols if col!='Electrical']
num_cols=[col for col in num_cols if col!='SalePrice']

In [362]:

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])


cat_mode_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('OHE',OneHotEncoder(handle_unknown='ignore'))
])

ohe_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('OHE',OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat_mode', cat_mode_pipeline, mod_col),
        ('OHE',ohe_pipeline,cat_cols)
    ],
    remainder='passthrough'
)

In [363]:
from sklearn.model_selection import GridSearchCV,KFold
from sklearn.metrics import r2_score,root_mean_squared_error
cv=KFold(n_splits=5,shuffle=True,random_state=42)

In [364]:
from sklearn.linear_model import LinearRegression,Lasso,Ridge,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,AdaBoostRegressor
reg=LinearRegression()
ls=Lasso()
rig=Ridge()
els=ElasticNet()
dt=DecisionTreeRegressor()
gbr=GradientBoostingRegressor()
rf=RandomForestRegressor()

In [365]:
param_grids = {

    'Linear Regression': {},

    'Ridge': {
        'alpha': [0.01,0.1,1,10,100]
    },

    'Lasso': {
        'alpha': [0.0001,0.001,0.01,0.1,1]
    },

    'ElasticNet': {
        'alpha':[0.0001,0.001,0.01,0.1,1],
        'l1_ratio':[0.2,0.5,0.7,0.9]
    },

    'Decision Tree': {
        'max_depth':[None,5,10,20],
        'min_samples_split':[2,5,10],
        'min_samples_leaf':[1,2,4]
    },

    'Random Forest': {
        'n_estimators':[100,200,300],
        'max_depth':[None,10,20],
        'min_samples_split':[2,5],
        'min_samples_leaf':[1,2]
    },

    'Gradient Boosting': {
        'n_estimators':[100,200],
        'learning_rate':[0.01,0.05,0.1],
        'max_depth':[3,4,5]
    },

    'AdaBoost': {
        'n_estimators':[50,100,200],
        'learning_rate':[0.01,0.1,1]
    }
}

param_grids = {
    model_name: {f"model__{k}": v for k, v in params.items()}
    for model_name, params in param_grids.items()
}

In [366]:
models = {
    'Linear Regression': reg,
    'Ridge': rig,
    'Lasso': ls,
    'ElasticNet': els,
    'Decision Tree': dt,
    'Random Forest': rf,
    'Gradient Boosting': gbr,
}

In [367]:
x=raw_data.drop(['SalePrice'],axis=1)
y = np.log1p(raw_data["SalePrice"])

In [368]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42)

In [369]:
best_score=-float('inf')
best_model=None
results=[]
for name,model in models.items():
    pipe=Pipeline([
    ('preprocessor',preprocessor),
    ('model',model)
    ])
    grid=GridSearchCV(estimator=pipe,param_grid=param_grids[name],cv=cv,scoring='r2',n_jobs=-1,error_score='raise')
    grid.fit(x_train,y_train)
    pred=grid.best_estimator_.predict(x_test)
    score=grid.best_score_
    if score>best_score:
        score=best_score
        best_model=grid.best_estimator_
    results.append({'Model':name,
                    'Best Params':grid.best_params_,
                    'CV score':grid.best_score_,
                    'R2':r2_score(y_test,pred),
                    'rmse':root_mean_squared_error(y_test,pred)})
results=pd.DataFrame(results)
results.sort_values(by='R2',ascending=False,inplace=True)

In [370]:
best_model.fit(x,y)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat_mode', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [371]:
test = feature_engineering(test)
test = ordinal_encode(test)

In [372]:
test2=pd.read_csv("test.csv")

In [373]:
predictions=best_model.predict(test)
predictions = np.expm1(predictions)

In [374]:
predictions

array([126343.82464257, 158688.31092393, 187725.11072308, ...,
       152314.38572974, 122854.79395928, 210701.40638089], shape=(1459,))

In [375]:
submission=pd.DataFrame({
    "ID":test2['Id'],
    "SalePrice":predictions
})

In [378]:
submission

,ID,SalePrice
0,1461,126343.824643
1,1462,158688.310924
2,1463,187725.110723
3,1464,178776.662390
4,1465,193233.355168
...,...,...
1454,2915,77695.961013
1455,2916,75109.515121
1456,2917,152314.385730
1457,2918,122854.793959


In [379]:
submission.to_csv("submission.csv",index=False)